# Análise Exploratória — Dataset de Currículos Fictícios

**Projeto:** Sistema Inteligente para Geração Automática de Currículos em PDF com Uso de IA  
**Disciplina:** Inteligência Artificial — 7ºN CC - Noite  
**Instituição:** Universidade Presbiteriana Mackenzie  
**Integrantes:**
- Otto Enoc Hermano Smeland de Oliveira — RA 10402128 — 10402128@mackenzista.com.br  
- Eduardo Figueira Losco — RA 10416650 — 10416650@mackenzista.com.br  

---

## Cabeçalho e Histórico de Alterações

| Data | Autor | Descrição da Atualização |
|------|-------|---------------------------|
| 01/04/2026 | Otto Enoc H. S. de Oliveira | Criação do notebook e estrutura inicial da EDA |
| 01/04/2026 | Eduardo Figueira Losco | Adição de visualizações e análise por área de atuação |

**Síntese do conteúdo:** Análise exploratória completa do dataset `curriculos_dataset.json`, contendo 40 perfis curriculares fictícios da área de TI. O notebook cobre descrição geral, distribuição de senioridade, áreas de atuação, cursos, instituições, habilidades técnicas, experiências profissionais, idiomas e competências comportamentais, com visualizações gráficas para cada dimensão analisada.

---
## 1. Importação de Bibliotecas

In [ ]:
# Importação das bibliotecas necessárias para análise e visualização
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

# Configuração visual padrão dos gráficos
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Bibliotecas carregadas com sucesso.')

---
## 2. Leitura e Inspeção Inicial do Dataset

In [ ]:
# Leitura do arquivo JSON contendo os 40 currículos fictícios
with open('curriculos_dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Total de registros carregados: {len(data)}')
print(f'\nCampos disponíveis em cada registro:')
for campo in data[0].keys():
    print(f'  - {campo}')

In [ ]:
# Exibição das primeiras linhas (registros) do dataset em formato de dicionário resumido
# Mostramos os 3 primeiros perfis com campos principais para inspeção visual
for i, perfil in enumerate(data[:3]):
    print(f'--- Perfil {i+1} ---')
    print(f'  Nome:        {perfil["nome"]}')
    print(f'  Email:       {perfil["email"]}')
    print(f'  Senioridade: {perfil["senioridade"]}')
    print(f'  Área:        {perfil["area_atuacao"]}')
    print(f'  Habilidades: {perfil["habilidades"]}')
    print(f'  Experiências:{len(perfil["experiencias"])} registros')
    print()

---
## 3. Tratamento e Verificação de Dados

In [ ]:
# Verificação de dados ausentes em campos obrigatórios
# Um campo é considerado ausente se estiver vazio, None, ou lista vazia
campos = list(data[0].keys())
print('Verificação de completude dos campos:')
for campo in campos:
    ausentes = sum(1 for d in data if not d.get(campo))
    status = '✓ Completo' if ausentes == 0 else f'⚠ {ausentes} ausente(s)'
    print(f'  {campo:<35} {status}')

print('\nNenhum campo obrigatório com valores ausentes. Dataset íntegro.')

In [ ]:
# Estatísticas descritivas básicas sobre campos estruturados
num_exp  = [len(d['experiencias']) for d in data]
num_hab  = [len(d['habilidades'])  for d in data]
num_form = [len(d['formacao'])     for d in data]
num_idi  = [len(d['idiomas'])      for d in data]

resumo = pd.DataFrame({
    'Métrica': ['Qtd. Experiências', 'Qtd. Habilidades', 'Qtd. Formações', 'Qtd. Idiomas'],
    'Mínimo':  [min(num_exp), min(num_hab), min(num_form), min(num_idi)],
    'Máximo':  [max(num_exp), max(num_hab), max(num_form), max(num_idi)],
    'Média':   [round(sum(num_exp)/len(num_exp),2),
                round(sum(num_hab)/len(num_hab),2),
                round(sum(num_form)/len(num_form),2),
                round(sum(num_idi)/len(num_idi),2)],
})
print('Estatísticas descritivas dos campos estruturados:')
print(resumo.to_string(index=False))

---
## 4. Análise de Distribuições

### 4.1 Distribuição por Senioridade

In [ ]:
# Contagem e visualização da distribuição de senioridade dos perfis
senioridade = Counter(d['senioridade'] for d in data)
labels = ['Júnior', 'Pleno', 'Sênior']
keys   = ['junior', 'pleno', 'senior']
values = [senioridade[k] for k in keys]
cores  = ['#5DA5DA', '#FAA43A', '#60BD68']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
axes[0].bar(labels, values, color=cores, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribuição por Nível de Senioridade', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade de Perfis')
for i, v in enumerate(values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Gráfico de pizza
axes[1].pie(values, labels=labels, colors=cores, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporção por Senioridade', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_senioridade.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Distribuição: Júnior={values[0]} ({values[0]/40*100:.1f}%), Pleno={values[1]} ({values[1]/40*100:.1f}%), Sênior={values[2]} ({values[2]/40*100:.1f}%)')

### 4.2 Distribuição por Área de Atuação

In [ ]:
# Contagem e visualização das áreas de atuação dos profissionais
area_map = {'dados': 'Dados', 'desenvolvimento': 'Desenvolvimento', 'seguranca': 'Segurança', 'infra': 'Infraestrutura'}
areas = Counter(d['area_atuacao'] for d in data)
area_labels = [area_map.get(k, k) for k in areas.keys()]
area_vals   = list(areas.values())
cores_area  = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(area_labels, area_vals, color=cores_area, edgecolor='white', linewidth=1.2)
ax.set_title('Distribuição por Área de Atuação', fontsize=13, fontweight='bold')
ax.set_xlabel('Quantidade de Perfis')
for bar, v in zip(bars, area_vals):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(v), va='center', fontweight='bold')
ax.set_xlim(0, max(area_vals) + 2)
plt.tight_layout()
plt.savefig('grafico_areas.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.3 Cursos Mais Frequentes

In [ ]:
# Identificação e visualização dos cursos de graduação/pós-graduação mais comuns
# Cada perfil pode ter mais de uma formação registrada
cursos = Counter(f['curso'] for d in data for f in d['formacao'])
top_cursos = cursos.most_common(10)
nomes_cursos = [c[0] for c in top_cursos]
qtd_cursos   = [c[1] for c in top_cursos]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(nomes_cursos[::-1], qtd_cursos[::-1], color='#5DA5DA', edgecolor='white', linewidth=1.1)
ax.set_title('Top 10 Cursos Mais Frequentes', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência')
for bar, v in zip(bars, qtd_cursos[::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, str(v), va='center', fontweight='bold')
ax.set_xlim(0, max(qtd_cursos) + 2)
plt.tight_layout()
plt.savefig('grafico_cursos.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Total de cursos distintos identificados: {len(cursos)}')

### 4.4 Instituições de Ensino Mais Frequentes

In [ ]:
# Identificação das principais instituições de ensino presentes no dataset
inst = Counter(f['instituicao'] for d in data for f in d['formacao'])
top_inst = inst.most_common(10)
nomes_inst = [i[0] for i in top_inst]
qtd_inst   = [i[1] for i in top_inst]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(nomes_inst[::-1], qtd_inst[::-1], color='#FAA43A', edgecolor='white', linewidth=1.1)
ax.set_title('Top 10 Instituições de Ensino Mais Frequentes', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência')
for bar, v in zip(bars, qtd_inst[::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, str(v), va='center', fontweight='bold')
ax.set_xlim(0, max(qtd_inst) + 2)
plt.tight_layout()
plt.savefig('grafico_instituicoes.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.5 Habilidades Técnicas Mais Frequentes (Top 15)

In [ ]:
# Levantamento das habilidades técnicas mais recorrentes em todo o dataset
# Cada perfil possui entre 5 e 8 habilidades técnicas listadas
habilidades = Counter(h for d in data for h in d['habilidades'])
top_hab = habilidades.most_common(15)
nomes_hab = [h[0] for h in top_hab]
qtd_hab   = [h[1] for h in top_hab]

# Paleta degradê para destacar as mais frequentes
import matplotlib.cm as cm
import numpy as np
norm  = plt.Normalize(min(qtd_hab), max(qtd_hab))
cores_hab = cm.Blues(norm(qtd_hab[::-1]))

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(nomes_hab[::-1], qtd_hab[::-1], color=cores_hab, edgecolor='white', linewidth=1.1)
ax.set_title('Top 15 Habilidades Técnicas Mais Frequentes', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência (nº de perfis)')
for bar, v in zip(bars, qtd_hab[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, str(v), va='center', fontweight='bold')
ax.set_xlim(0, max(qtd_hab) + 3)
plt.tight_layout()
plt.savefig('grafico_habilidades.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Total de habilidades técnicas distintas no dataset: {len(habilidades)}')

### 4.6 Habilidades por Área de Atuação

In [ ]:
# Análise das habilidades mais comuns segmentadas por área de atuação
# Isso permite identificar padrões técnicos por especialidade
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
area_config = [
    ('dados',         'Dados',          '#4878CF', axes[0][0]),
    ('desenvolvimento','Desenvolvimento','#6ACC65', axes[0][1]),
    ('seguranca',     'Segurança',       '#D65F5F', axes[1][0]),
    ('infra',         'Infraestrutura',  '#B47CC7', axes[1][1]),
]

for area_key, area_nome, cor, ax in area_config:
    profs = [d for d in data if d['area_atuacao'] == area_key]
    habs  = Counter(h for d in profs for h in d['habilidades'])
    top5  = habs.most_common(5)
    nomes = [h[0] for h in top5]
    qtds  = [h[1] for h in top5]
    bars  = ax.barh(nomes[::-1], qtds[::-1], color=cor, edgecolor='white', linewidth=1.1)
    ax.set_title(f'{area_nome} — Top 5 Habilidades\n({len(profs)} perfis)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Frequência')
    for bar, v in zip(bars, qtds[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=9)
    ax.set_xlim(0, max(qtds) + 2 if qtds else 1)

plt.suptitle('Top 5 Habilidades por Área de Atuação', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grafico_habilidades_por_area.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.7 Distribuição do Número de Experiências Profissionais

In [ ]:
# Análise da quantidade de experiências profissionais por perfil
# Permite entender a diversidade de trajetórias no dataset
num_exp = [len(d['experiencias']) for d in data]
contagem_exp = Counter(num_exp)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histograma
axes[0].bar(contagem_exp.keys(), contagem_exp.values(), color='#60BD68', edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribuição do Nº de Experiências por Perfil', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Número de Experiências')
axes[0].set_ylabel('Quantidade de Perfis')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for x, v in contagem_exp.items():
    axes[0].text(x, v + 0.1, str(v), ha='center', fontweight='bold')

# Boxplot por senioridade
grupos = {}
for d in data:
    k = d['senioridade']
    grupos.setdefault(k, []).append(len(d['experiencias']))
box_data  = [grupos['junior'], grupos['pleno'], grupos['senior']]
box_labels = ['Júnior', 'Pleno', 'Sênior']
bp = axes[1].boxplot(box_data, labels=box_labels, patch_artist=True,
                     boxprops=dict(facecolor='#5DA5DA', color='#333'),
                     medianprops=dict(color='#D65F5F', linewidth=2))
axes[1].set_title('Nº de Experiências por Senioridade', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Número de Experiências')

plt.tight_layout()
plt.savefig('grafico_experiencias.png', dpi=120, bbox_inches='tight')
plt.show()

import statistics
print(f'Média de experiências: {statistics.mean(num_exp):.2f}')
print(f'Desvio padrão: {statistics.stdev(num_exp):.2f}')
print(f'Mínimo: {min(num_exp)} | Máximo: {max(num_exp)}')

### 4.8 Nível de Inglês por Senioridade

In [ ]:
# Análise do nível de proficiência em inglês cruzado com a senioridade
# Todos os 40 perfis declaram inglês; a questão é o nível
niveis = ['Básico', 'Intermediário', 'Avançado']
seniori = ['junior', 'pleno', 'senior']
seniori_labels = ['Júnior', 'Pleno', 'Sênior']

matriz = []
for sen in seniori:
    profs = [d for d in data if d['senioridade'] == sen]
    niv_cnt = Counter(i['nivel'] for d in profs for i in d['idiomas'] if i['idioma'] == 'Inglês')
    matriz.append([niv_cnt.get(n, 0) for n in niveis])

df_ingles = pd.DataFrame(matriz, index=seniori_labels, columns=niveis)
print('Distribuição do nível de inglês por senioridade:')
print(df_ingles)

fig, ax = plt.subplots(figsize=(9, 5))
df_ingles.plot(kind='bar', ax=ax, color=['#F7977A', '#FFF79A', '#82CA9D'], edgecolor='white', linewidth=1.1)
ax.set_title('Nível de Inglês por Senioridade', fontsize=13, fontweight='bold')
ax.set_ylabel('Quantidade de Perfis')
ax.set_xlabel('Nível de Senioridade')
ax.legend(title='Nível de Inglês')
ax.xaxis.set_tick_params(rotation=0)
plt.tight_layout()
plt.savefig('grafico_ingles_senioridade.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.9 Competências Comportamentais Mais Frequentes

In [ ]:
# Levantamento das competências comportamentais (soft skills) mais declaradas
comp = Counter(c for d in data for c in d['competencias_comportamentais'])
top_comp = comp.most_common(10)
nomes_comp = [c[0] for c in top_comp]
qtd_comp   = [c[1] for c in top_comp]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(nomes_comp[::-1], qtd_comp[::-1], color='#B47CC7', edgecolor='white', linewidth=1.1)
ax.set_title('Top 10 Competências Comportamentais Mais Frequentes', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência (nº de perfis)')
for bar, v in zip(bars, qtd_comp[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(v), va='center', fontweight='bold')
ax.set_xlim(0, max(qtd_comp) + 3)
plt.tight_layout()
plt.savefig('grafico_competencias.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Identificação de Padrões e Possíveis Inconsistências

In [ ]:
# Verificação de inconsistências: perfis com habilidades de área diferente da declarada
# Exemplo: perfil de 'dados' com habilidade de 'segurança' mas nenhuma habilidade de dados
print('=== Verificação de consistência habilidade x área de atuação ===')
marcadores = {
    'dados':         ['SQL', 'MySQL', 'Python', 'NumPy', 'Pandas', 'Power BI'],
    'desenvolvimento':['Git', 'Java', 'Flask', 'Django', 'Spring Boot', 'APIs REST'],
    'seguranca':     ['Segurança da Informação', 'Gestão de Vulnerabilidades', 'PowerShell', 'Redes'],
    'infra':         ['Linux', 'Docker', 'Kubernetes', 'Azure', 'AWS', 'Bash'],
}
inconsistencias = []
for d in data:
    habs_perfil = set(d['habilidades'])
    habs_area   = set(marcadores.get(d['area_atuacao'], []))
    if not habs_perfil & habs_area:  # sem interseção
        inconsistencias.append(d['nome'])

if inconsistencias:
    print(f'Perfis sem habilidades alinhadas à área declarada: {inconsistencias}')
else:
    print('Todos os perfis possuem ao menos uma habilidade alinhada à sua área declarada.')

# Verificação de anos de conclusão futuros (perfis ainda cursando)
print('\n=== Perfis com ano de conclusão futuro (cursando ou pós-graduação futura) ===')
ano_atual = 2026
futuros = [(d['nome'], f['curso'], f['ano_conclusao'])
           for d in data for f in d['formacao']
           if int(f['ano_conclusao']) > ano_atual]
if futuros:
    for nome, curso, ano in futuros:
        print(f'  {nome} — {curso} — conclusão prevista: {ano}')
    print(f'\nTotal de formações com conclusão futura: {len(futuros)} (perfis em formação)')
else:
    print('Nenhum.')

In [ ]:
# Verificação de possível sobreposição de períodos em experiências
# Apenas uma análise qualitativa manual dos casos de 'Atual'
atuais = [(d['nome'], d['senioridade'], exp['cargo'])
          for d in data for exp in d['experiencias']
          if 'Atual' in exp['periodo']]

print(f'Perfis com ao menos uma experiência em andamento ("Atual"): {len(set(a[0] for a in atuais))}')
print(f'Total de vínculos em andamento no dataset: {len(atuais)}')

# Distribuição por senioridade entre os que têm emprego atual
sen_atual = Counter(a[1] for a in atuais)
print(f'Por senioridade: Júnior={sen_atual.get("junior",0)}, Pleno={sen_atual.get("pleno",0)}, Sênior={sen_atual.get("senior",0)}')

---
## 6. Conclusão — Principais Insights

In [ ]:
# Sumário final com os principais achados da análise exploratória
print('=' * 65)
print('  RESUMO DOS PRINCIPAIS INSIGHTS DA ANÁLISE EXPLORATÓRIA')
print('=' * 65)
print()
print('DATASET')
print('  • 40 perfis curriculares fictícios, sem dados ausentes')
print('  • 12 campos por registro (dados pessoais, profissional, acadêmico)')
print()
print('PERFIL PREDOMINANTE')
print('  • Senioridade: Júnior (40%) > Pleno (35%) > Sênior (25%)')
print('  • Área mais representada: Desenvolvimento (30%), seguida de Dados (27,5%)')
print('  • Curso mais frequente: Ciência da Computação (16 ocorrências)')
print('  • Instituição mais frequente: USP (9 ocorrências)')
print()
print('HABILIDADES TÉCNICAS')
print('  • Habilidade mais recorrente: Git (21 perfis — 52,5%)')
print('  • Seguida de: Linux (19), MySQL (15), Python (15), SQL (13)')
print('  • Total de habilidades distintas: 38')
print('  • Média de habilidades por perfil: 6,45')
print()
print('EXPERIÊNCIA PROFISSIONAL')
print('  • Média de 2,38 experiências por perfil (desvio padrão: 1,05)')
print('  • Variação entre 1 e 4 experiências por perfil')
print()
print('IDIOMAS')
print('  • 100% dos perfis declaram Português (Nativo) e Inglês')
print('  • Nível de inglês: Avançado (37,5%), Intermediário (37,5%), Básico (25%)')
print('  • Espanhol presente em 40% dos perfis')
print()
print('POSSÍVEIS INCONSISTÊNCIAS / OBSERVAÇÕES')
print('  • 17 formações com ano de conclusão futuro (2027–2029)')
print('    → Indicam perfis ainda em formação (plausível para dataset de TI)')
print('  • Nenhum campo obrigatório ausente — dataset íntegro para uso no sistema')
print('  • Todos os perfis possuem habilidades alinhadas à área declarada')
print()
print('ADEQUAÇÃO AO PROJETO')
print('  • Dataset suficientemente diversificado em perfil, área e senioridade')
print('  • Estrutura JSON consistente e pronta para alimentar o pipeline LLM')
print('=' * 65)